# Load and Prepare Flow Measurements

Read the per-patient flow measurements CSV (Ascending Aorta and Main PA),
clean the data, reshape into long format, compute error metrics, and save a
consolidated CSV for downstream analysis.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

NOTEBOOK_DIR = Path.home() / "vascular-superenhancement-4d-flow" / "notebooks" / "analysis" / "PEC_quantitative_analysis"
CSV_PATH = NOTEBOOK_DIR / "Cardiac 3D Cine _ 4D Flow - test-population_flow-measurements_albert.csv"
OUTPUT_CSV = NOTEBOOK_DIR / "consolidated.csv"

assert CSV_PATH.exists(), f"CSV not found: {CSV_PATH}"

## Load raw CSV

In [2]:
raw = pd.read_csv(CSV_PATH)
print(f"{len(raw)} patients")
print(f"Columns: {list(raw.columns)}")
raw

15 patients
Columns: ['Original PID [UCSD - Radiology]', 'CNN Vascular Enhanced + Phase Error Corrected PID [UCSD - CTIPM]', 'Manually Corrected Ascending Ao (Flow)', 'Manually Corrected Main PA (Flow)', 'Uncorrected Ascending Ao (Flow)', 'Uncorrected Main PA (Flow)', 'CNN Corrected Ascending Ao (Flow)', 'CNN Corrected Main PA (Flow)', 'Comments']


,Original PID [UCSD - Radiology],CNN Vascular Enhanced + Phase Error Corrected PID [UCSD - CTIPM],Manually Corrected Ascending Ao (Flow),Manually Corrected Main PA (Flow),Uncorrected Ascending Ao (Flow),Uncorrected Main PA (Flow),CNN Corrected Ascending Ao (Flow),CNN Corrected Main PA (Flow),Comments
0,Balboloop,Joshcefoey,6.32,6.52,6.92,7.63,6.73,6.61,NaN
1,Biswifo,Ilumid,5.67,5.91,6.91,5.05,5.66,5.70,NaN
2,Bomatog,Geseyar,7.53,6.70,6.74,5.75,6.91,6.22,NaN
3,Detodu,Cuhodud,4.50,4.44,5.40,3.60,5.43,4.65,NaN
4,Diecudey,Strerofri,7.46,7.18,9.14,6.12,7.44,5.66,NaN
5,Diepami,Edeshcey,4.82,4.62,6.73,7.20,5.84,5.99,NaN
6,Diequipi,Yokrobe,4.48,4.26,6.25,5.98,4.36,4.35,NaN
7,Dublafer,Ergoopey,5.90,6.32,6.35,7.41,6.06,6.32,good CNN correction
8,Dujomal,Jeface,9.85,8.54,10.72,7.89,10.06,8.92,NaN
9,Golotag,Yeequesu,4.16,11.36,3.98,10.36,4.75,13.02,shunt / papvr


## Reshape to long format

Target schema: `patient | vessel | method | flow | comments`

- **vessel**: `Ao` (Ascending Aorta) or `PA` (Main Pulmonary Artery)
- **method**: `manual`, `uncorrected`, `cnn`

In [3]:
COL_MAP = {
    ("Ao", "manual"): "Manually Corrected Ascending Ao (Flow)",
    ("Ao", "uncorrected"): "Uncorrected Ascending Ao (Flow)",
    ("Ao", "cnn"): "CNN Corrected Ascending Ao (Flow)",
    ("PA", "manual"): "Manually Corrected Main PA (Flow)",
    ("PA", "uncorrected"): "Uncorrected Main PA (Flow)",
    ("PA", "cnn"): "CNN Corrected Main PA (Flow)",
}

frames = []
for (vessel, method), col in COL_MAP.items():
    sub = raw[["Original PID [UCSD - Radiology]", col, "Comments"]].copy()
    sub.columns = ["patient", "flow", "comments"]
    sub["vessel"] = vessel
    sub["method"] = method
    frames.append(sub)

data = pd.concat(frames, ignore_index=True)
data = data[["patient", "vessel", "method", "flow", "comments"]]

print(f"{len(data)} rows  |  {data['patient'].nunique()} patients  |  "
      f"vessels: {sorted(data['vessel'].unique())}  |  methods: {sorted(data['method'].unique())}")
data.head(9)

90 rows  |  15 patients  |  vessels: ['Ao', 'PA']  |  methods: ['cnn', 'manual', 'uncorrected']


,patient,vessel,method,flow,comments
0,Balboloop,Ao,manual,6.32,NaN
1,Biswifo,Ao,manual,5.67,NaN
2,Bomatog,Ao,manual,7.53,NaN
3,Detodu,Ao,manual,4.50,NaN
4,Diecudey,Ao,manual,7.46,NaN
5,Diepami,Ao,manual,4.82,NaN
6,Diequipi,Ao,manual,4.48,NaN
7,Dublafer,Ao,manual,5.90,good CNN correction
8,Dujomal,Ao,manual,9.85,NaN


## Handle missing data

In [4]:
print("Missing flow values:")
missing = data[data["flow"].isna()]
if len(missing):
    print(missing[["patient", "vessel", "method"]].to_string(index=False))
else:
    print("  None")

print(f"\nPatients with comments:")
commented = raw[raw["Comments"].notna()][["Original PID [UCSD - Radiology]", "Comments"]]
commented.columns = ["patient", "comments"]
print(commented.to_string(index=False))

Missing flow values:
 patient vessel      method
Oduskueb     PA      manual
Oduskueb     PA uncorrected
Oduskueb     PA         cnn

Patients with comments:
 patient                 comments
Dublafer      good CNN correction
 Golotag            shunt / papvr
Gueshifa      good CNN correction
Oduskueb                   fontan
Suquepog vsd, good CNN correction
Tiepolem              shunt / PDA


## Compute error metrics

For each patient and vessel, compute the error of `uncorrected` and `cnn` methods
relative to `manual` (ground truth):
- **error** = method − manual (signed)
- **abs_error** = |method − manual|
- **pct_error** = (method − manual) / manual × 100

In [5]:
manual = data[data["method"] == "manual"][["patient", "vessel", "flow"]].rename(
    columns={"flow": "manual_flow"}
)

compared = data[data["method"].isin(["uncorrected", "cnn"])].merge(
    manual, on=["patient", "vessel"], how="inner"
)

compared["error"] = compared["flow"] - compared["manual_flow"]
compared["abs_error"] = compared["error"].abs()
compared["pct_error"] = (compared["error"] / compared["manual_flow"]) * 100

print(f"{len(compared)} comparison rows")
compared.head()

60 comparison rows


,patient,vessel,method,flow,comments,manual_flow,error,abs_error,pct_error
0,Balboloop,Ao,uncorrected,6.92,NaN,6.32,0.60,0.60,9.493671
1,Biswifo,Ao,uncorrected,6.91,NaN,5.67,1.24,1.24,21.869489
2,Bomatog,Ao,uncorrected,6.74,NaN,7.53,-0.79,0.79,-10.491368
3,Detodu,Ao,uncorrected,5.40,NaN,4.50,0.90,0.90,20.000000
4,Diecudey,Ao,uncorrected,9.14,NaN,7.46,1.68,1.68,22.520107


## Sanity checks

In [6]:
print("Flow value ranges per method:")
print(data.groupby("method")["flow"].describe().round(2))
print()

pivot_check = data.pivot_table(index=["patient", "vessel"], columns="method", values="flow")
n_complete = pivot_check.dropna().shape[0]
n_total = pivot_check.shape[0]
print(f"Complete cases (all 3 methods): {n_complete}/{n_total}")

print("\nError summary by method and vessel:")
print(compared.groupby(["vessel", "method"])[["error", "abs_error", "pct_error"]]
      .agg(["mean", "std"]).round(2))

Flow value ranges per method:
             count  mean   std   min   25%   50%   75%    max
method                                                       
cnn           29.0  6.33  2.01  2.82  5.05  6.06  6.91  13.02
manual        29.0  6.15  1.85  3.15  4.62  6.32  7.18  11.36
uncorrected   29.0  6.77  1.71  3.60  5.75  6.74  7.72  10.72

Complete cases (all 3 methods): 29/29

Error summary by method and vessel:
                   error       abs_error       pct_error       
                    mean   std      mean   std      mean    std
vessel method                                                  
Ao     cnn          0.18  0.44      0.35  0.31      3.91   8.88
       uncorrected  0.76  0.92      1.05  0.54     14.54  18.81
PA     cnn          0.18  0.77      0.54  0.56      2.27  11.84
       uncorrected  0.45  1.34      1.22  0.64     12.10  29.10


## Save consolidated data

In [7]:
NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)

data.to_csv(OUTPUT_CSV, index=False)
print(f"Saved long-format data: {OUTPUT_CSV}")

COMPARED_CSV = NOTEBOOK_DIR / "error_metrics.csv"
compared.to_csv(COMPARED_CSV, index=False)
print(f"Saved error metrics: {COMPARED_CSV}")

Saved long-format data: /home/ayeluru/vascular-superenhancement-4d-flow/notebooks/analysis/PEC_quantitative_analysis/consolidated.csv
Saved error metrics: /home/ayeluru/vascular-superenhancement-4d-flow/notebooks/analysis/PEC_quantitative_analysis/error_metrics.csv
